In [1]:
# =========================================================
# LIGHTWEIGHT CONVERSATIONAL AI
# DialoGPT + Memory System
# =========================================================

# =========================
# 1. IMPORTS
# =========================

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

# =========================
# 2. LOAD TOKENIZER
# =========================

tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/DialoGPT-small"
)

tokenizer.pad_token = tokenizer.eos_token

# =========================
# 3. LOAD MODEL
# =========================

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/DialoGPT-small"
)

# =========================
# 4. DEVICE
# =========================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

print("Model Loaded Successfully!")

# =========================
# 5. MEMORY SYSTEM
# =========================

BOT_PERSONA = """
Your name is Bryse.
You are a friendly AI assistant.
You speak clearly and calmly.
You like technology and conversations.
"""

conversation_history = ""

memory = {}

# =========================
# 6. CHAT FUNCTION
# =========================

def chat(text):

    global conversation_history
    global memory

    model.eval()

    # =========================
    # SAVE NAME
    # =========================

    if "my name is" in text.lower():

        name = text.split("is")[-1].strip()

        memory["name"] = name

    # =========================
    # SAVE LIKES
    # =========================

    if text.lower().startswith("i like"):

        like = text[6:].strip()

        memory["likes"] = like

    # =========================
    # MEMORY RESPONSES
    # =========================

    if "what is my name" in text.lower():

        if "name" in memory:

            return f"Your name is {memory['name']}"

        else:

            return "I don't know your name yet."

    if "what do i like" in text.lower():

        if "likes" in memory:

            return f"You like {memory['likes']}"

        else:

            return "I don't know what you like yet."

    # =========================
    # BUILD CONTEXT
    # =========================

    conversation_history += f"User: {text}\n"

    input_text = BOT_PERSONA + "\n" + conversation_history + "Bot:"

    # =========================
    # TOKENIZE
    # =========================

    inputs = tokenizer.encode(
        input_text + tokenizer.eos_token,
        return_tensors="pt"
    ).to(device)

    # =========================
    # GENERATE RESPONSE
    # =========================

    outputs = model.generate(
        inputs,
        max_new_tokens=30,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        temperature=0.5,
        top_k=20,
        top_p=0.95
    )

    # =========================
    # DECODE
    # =========================

    response = tokenizer.decode(
        outputs[:, inputs.shape[-1]:][0],
        skip_special_tokens=True
    )

    # =========================
    # SAVE RESPONSE
    # =========================

    conversation_history += f"Bot: {response}\n"

    response = response.replace("Bot :", "")
    response = response.strip()

    return response

# =========================
# 7. TEST
# =========================

print("\n====================")
print("CHAT TEST")
print("====================\n")

print("User: Hello")
print("Bot :", chat("Hello"))

print()

print("User: My name is Ahmed")
print("Bot :", chat("My name is Ahmed"))

print()

print("User: What is my name?")
print("Bot :", chat("What is my name?"))

print()

print("User: I like football")
print("Bot :", chat("I like football"))

D:\Anaconda3\envs\chatbot_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████████████████████████████████████████████| 149/149 [00:00<00:00, 3358.62it/s]
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Model Loaded Successfully!

CHAT TEST

User: Hello


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Bot : I like your username .

User: My name is Ahmed
Bot : My name is Ahmed

User: What is my name?
Bot : Your name is Ahmed

User: I like football
Bot : I like the way you think .
